In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import csv
import time
import requests
from bs4 import BeautifulSoup
import re
import json
import os
from urllib.parse import urlparse, parse_qs

In [23]:
headers = {
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win 64; x64) AppleWebKit/537.36 (KHTML, Like Gecko) Chrome/140.0.0.0 Safari/537.36 Edg/140.0.0.0'
}

folder = 'data_berita'

# SCRAPING CNBC INDONESIA
link searchnya: https://www.cnbcindonesia.com/search?query=politik+pemerintah+indonesia&fromdate=2025/08/25&todate=2025/09/25

In [24]:
# baca link berita
def get_article(link):
  try:
    res = requests.get(link, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    div_content = soup.find('div', class_='detail-text') #ganti class sesuai nama class di artikel
    paragraphs = div_content.find_all('p') #ganti tiap tag paragraf
    content = ' '.join([p.get_text(strip=True) for p in paragraphs])
    return content
    
  except Exception as e:
    print(f"Error artikel di: {link} | {e}")
  return ''


In [25]:
# url list
urlcnbc = 'https://www.cnbcindonesia.com/search?query=politik+indonesia&fromdate=2015/01/31&todate=2025/09/30'

rescnbc = requests.get(urlcnbc, headers=headers)
soupcnbc = BeautifulSoup(rescnbc.text, 'lxml')

ProxyError: HTTPSConnectionPool(host='www.cnbcindonesia.com', port=443): Max retries exceeded with url: /search?query=politik+indonesia&fromdate=2015/01/31&todate=2025/09/30 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 407 Proxy Authentication Required')))

In [26]:
pagenumbers = []
for a in soupcnbc.find_all('a'): #'a' bisa diganti, cek tag tag an di max page hsil search
  try:
    pagenumbers.append(int(a.get_text()))
  except:
    continue

lastpage = max(pagenumbers)
print(f"Halaman yg bisa di scraping: {lastpage}")

Halaman yg bisa di scraping: 834


In [31]:
#FILE DRIVERS

#create folder buat store data
if not os.path.exists(folder):
    os.makedirs(folder)


In [ ]:
# SCRAPING

#writer untuk masuk ke csv
with open(f'{folder}/cnbc_scraping_politik_article.csv', 'w', newline='', encoding='utf-8', buffering=1) as f:
  writer = csv.writer(f)
  writer.writerow(['Title', 'Link', 'Time', 'Content'])


counter = 0
temp = -100
# main function to scrap
for page in range(1, lastpage+1):
  print(f'scraping hlmn ke - {page}')
  url = f'https://www.cnbcindonesia.com/search?query=politik+pemerintah+indonesia&page={page}&fromdate=2024/12/25&todate=2025/01/01' #test, dari 25 des 2024 smpe 1 jan 2025
  try:
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    articles = soup.find_all('article') #sesuaiin ke website 

    for art in articles:
      try:

        a_tag = art.find('a') #sesuaiin ke website (di cnbc ini ngebungkus berita, kek header gt)
        title = a_tag.find('h2').text.strip() #sesuaiin ke website (di cnbc ini tag tag an buat judul)
        link = a_tag['href'] #sesuaiin ke website (di cnbc ini buat tembak site beritanya)
        time_info = a_tag.select_one('span')[-1].text.strip() #sesuaiin ke website (di cnbc ini tag untuk )
        content = get_article(link)

        # save ke csv
        # with open(path, 'a', newline='', encoding='utf-8') as f:
        #   writer = csv.writer(f)
        with open(f'{folder}/cnbc_scraping_politik_article.csv', 'a', newline='', encoding='utf-8', buffering=1) as f:
          writer = csv.writer(f)
          writer.writerow([title, link, time_info, content]) 
        # f.flush()
        print('masuk')

        print(f'✅ {title} SCRAPPED!')
        print(f'    100 KARAKTER PERTAMA DI CONTENT: {content[:100]}\n waktunya: {time_info}')
        counter += 1
        temp += 1
        time.sleep(1)
        if temp == 0 :
          time.sleep(59)
          temp = -100
      
      except Exception as e:
        print(f"Error parsing artikel di hlman: {page} | {e}")
        continue
  except Exception as e:
    print(f"Error scraping di hlman: {page} | {e}")
    continue

scraping hlmn ke - 1
Error artikel di: https://www.cnbcindonesia.com/lifestyle/20250101120843-33-600015/warga-ri-berbondong-bondong-pindah-jadi-wn-singapura-bayar-berapa | 'NoneType' object has no attribute 'find_all'
masuk
✅ Warga RI Berbondong-bondong Pindah Jadi WN Singapura, Bayar Berapa? SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: 
 waktunya: 8 bulan yang lalu
Error artikel di: https://www.cnbcindonesia.com/research/20250101100258-128-600006/tahun-baru-di-korsel-penuh-duka-tragedi-jeju-hingga-kospi-merona | 'NoneType' object has no attribute 'find_all'
masuk
✅ Tahun Baru di Korsel Penuh Duka: Tragedi Jeju Hingga Kospi Merona SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: 
 waktunya: 8 bulan yang lalu
Error artikel di: https://www.cnbcindonesia.com/market/20250101073314-17-599990/anggota-tni-mendadak-dapat-rp-50-miliar-semalam-langsung-jadi-okb | 'NoneType' object has no attribute 'find_all'
masuk
✅ Anggota TNI Mendadak Dapat Rp 50 Miliar Semalam, Langsung Jadi OKB SCRAPPED!
   

KeyboardInterrupt: 

In [9]:
print(f'artikel keambil : {counter} artikel')

artikel keambil : 35 artikel


# SCRAPING KOMPAS
link searchnya : https://search.kompas.com/search?q=politik+pemerintah+indonesia&sort=latest&site_id=1&last_date=all

In [27]:
# baca link berita
def get_article(link):
  try:
    res = requests.get(link, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    div_content = soup.find('div', class_='read__content') #ganti class sesuai nama class di artikel
    paragraphs = div_content.find_all('p') #ganti tiap tag paragraf
    content = ' '.join([p.get_text(strip=True) for p in paragraphs])
    return content
    
  except Exception as e:
    print(f"Error artikel di: {link} | {e}")
  return ''


In [29]:
# url list
urlkompas = 'https://search.kompas.com/search?q=politik+pemerintah+indonesia&sort=latest&site_id=1&last_date=all'
respkompas = requests.get(urlkompas)
soupkompas = BeautifulSoup(respkompas.text, 'lxml')

# print(soupkompas.text)


In [30]:
last_page_kompas = None

# 1) coba langsung selector 'Last' (class paging__link--last)
last_a = soupkompas.select_one('a.paging__link--last')

def extract_page_from_href(href):
    if not href:
        return None
    # coba regex ?page=123 atau &page=123
    m = re.search(r'[?&]page=(\d+)', href)
    if m:
        return int(m.group(1))
    # fallback parse query
    qs = parse_qs(urlparse(href).query)
    if 'page' in qs and qs['page']:
        try:
            return int(qs['page'][0])
        except ValueError:
            return None
    return None

if last_a:
    # prioritas: data-ci-pagination-page attribute
    data_page = last_a.get('data-ci-pagination-page')
    if data_page and data_page.isdigit():
        last_page_kompas = int(data_page)
    else:
        # extract dari href
        last_page_kompas = extract_page_from_href(last_a.get('href'))

print("Halaman terakhir terdeteksi:", last_page_kompas)

Halaman terakhir terdeteksi: 499


In [ ]:
# scraping

#writer untuk masuk ke csv
with open(f'{folder}/kompas_scraping_politik_article.csv', 'w', newline='', encoding='utf-8', buffering=1) as f:
  writer = csv.writer(f)
  writer.writerow(['Title', 'Link', 'Time', 'Content'])


counter = 0
#disinii gw bikin page 2 aja, biar ga banyak kan testing, harusnya variable 'testpage' dibawah ganti ke 'last_page_kompas'
for page in range(1, testpage+1):
  print(f'scraping hlmn ke - {last_page_kompas}')
  url = f'https://search.kompas.com/search?q=politik+pemerintah+indonesia&sort=latest&site_id=1&last_date=all&page={page}' #test, dari 25 des 2024 smpe 1 jan 2025
  try:
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    articles = soup.find_all('a', class_='article-link') #sesuaiin ke website 

    for art in articles:
      try:

        # a_tag = art.find('a') #sesuaiin ke website (di kompas ini ngebungkus berita, kek header gt)
        title = art.find('h2').text.strip() #sesuaiin ke website (di kompas ini tag tag an buat judul)
        link = art['href'] #sesuaiin ke website (di cnbc ini buat tembak site beritanya)
        time_info = art.find('div', class_='articlePost-date').text.strip() #sesuaiin ke website (di cnbc ini tag untuk )
        content = get_article(link)

        # save ke csv
        with open(f'{folder}/kompas_scraping_politik_article.csv', 'a', newline='', encoding='utf-8', buffering=1) as f:
          writer = csv.writer(f)
          writer.writerow([title, link, time_info, content])

        print(f'✅ {title} SCRAPPED!')
        print(f'    100 KARAKTER PERTAMA DI CONTENT: {content[:100]} \n waktunya: {time_info}')
        counter += 1
        time.sleep(1)
      
      except Exception as e:
        print(f"Error parsing artikel di hlman: {page} | {e}")
        continue

        time.sleep(1)
  except Exception as e:
    print(f"Error scraping di hlman: {page} | {e}")
    continue

scraping hlmn ke - 1
✅ Negara Anggota BRICS Plus Kompak Dukung Palestina, Indonesia Ucapkan Terima Kasih SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA, KOMPAS.com- Menteri Luar Negeri RI, Sugiono, menggelar pertemuan bilateral dengan negara-ne 
 waktunya: 27 September 2025
✅ Indonesia Tegaskan Posisi di Kancah Global Lewat Kesepakatan IEU–CEPA dan ICA–CEPA SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: KOMPAS.com –Selepas perundingan panjang selama beberapa tahun, Indonesia akhirnya menandatangani dua 
 waktunya: 27 September 2025
✅ Buka Jak-Japan Matsuri, Pramono: Hubungan Indonesia-Jepang Harmonis tanpa Konflik Politik SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA, KOMPAS.com –Gubernur DKI Jakarta Pramono Anung menegaskan, Jakarta merupakan rumah bagi ber 
 waktunya: 27 September 2025
✅ Prabowo: Belanda Kembalikan 30.000 Artefak ke Indonesia SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA, KOMPAS.com- Presiden Prabowo Subianto mengungkapkan bahwa Belanda akan

# SCRAPING CNN INDONESIA
link: https://www.cnnindonesia.com/search?query=politik+pemerintah+indonesia&kanal=nasional&fromdate=01%2F12%2F2024&todate=01%2F01%2F2025

In [1]:
# baca link berita
def get_article(link):
  try:
    res = requests.get(link, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    div_content = soup.find('div', class_='read__content') #ganti class sesuai nama class di artikel
    paragraphs = div_content.find_all('p') #ganti tiap tag paragraf
    content = ' '.join([p.get_text(strip=True) for p in paragraphs])
    return content
    
  except Exception as e:
    print(f"Error artikel di: {link} | {e}")
  return ''


In [9]:
# url list
urlcnn = 'https://www.cnnindonesia.com/search?query=politik+pemerintah+indonesia&kanal=nasional&fromdate=01%2F12%2F2024&todate=01%2F01%2F2025'
respcnn = requests.get(urlcnn)
soupcnn = BeautifulSoup(respcnn.text, 'lxml')

# print(soupcnn.text)


In [8]:

page_links = soupcnn.select('a[href*="page="]')
lastpage_cnn = -1
max_link = None
for a in page_links:
    m = re.search(r'page=(\d+)', a.get('href', ''))
    if m:
        p = int(m.group(1))
        if p > lastpage_cnn:
            lastpage_cnn = p
            max_link = a

# hasil:
if max_link:
    print(lastpage_cnn, max_link.get_text(strip=True), max_link.get('href'))


print("Halaman terakhir terdeteksi:", lastpage_cnn)

Halaman terakhir terdeteksi: -1


# SCRAPING DETIK.COM

In [12]:
urldetik = 'https://www.detik.com/search/searchnews?query=politik%20pemerintah%20indonesia&result_type=relevansi&fromdatex=01/12/2024&todatex=15/01/2025'

In [14]:
respdetik = requests.get(urldetik, headers=headers)
soupdetik = BeautifulSoup(respdetik.text, 'lxml')

print(soupdetik)

ProxyError: HTTPSConnectionPool(host='www.detik.com', port=443): Max retries exceeded with url: /search/searchnews?query=politik%20pemerintah%20indonesia&result_type=relevansi&fromdatex=01/12/2024&todatex=15/01/2025 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 504 Gateway Timeout')))